In [1]:
#!/usr/bin/env python3
import argparse
import os
import numpy as np
import matplotlib.pyplot as plt

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--train_txt", type=str, required=True,
                   help="Path to train.txt containing lines like 'data/XXXXX.npy <label>'")
    p.add_argument("--data_root", type=str, default=".",
                   help="Root folder so that os.path.join(data_root, relpath) points to the .npy files")
    p.add_argument("--fs", type=float, default=62_500_000.0,
                   help="Sample rate in Hz (default 62.5 MHz from dataset README)")
    p.add_argument("--nfft", type=int, default=1024, help="FFT size for spectrogram")
    p.add_argument("--hop", type=int, default=256, help="Hop size (overlap = nfft - hop)")
    p.add_argument("--win", type=str, default="hann", help="Window type: hann, hamming, boxcar")
    p.add_argument("--max_items", type=int, default=10, help="How many entries to visualize")
    p.add_argument("--vmin_db", type=float, default=None, help="Optional lower dB clip for imshow")
    p.add_argument("--vmax_db", type=float, default=None, help="Optional upper dB clip for imshow")
    return p.parse_args()

def load_iq(path):
    """
    Load IQ from .npy with a few common layouts:
      - complex dtype: shape (N,)
      - real 2-col: shape (N,2) -> I + jQ
      - interleaved 1D real: shape (2N,) -> I,Q,I,Q,... (will try to interpret)
    Returns complex64 array shape (N,).
    """
    arr = np.load(path, allow_pickle=False)
    # If already complex
    if np.iscomplexobj(arr):
        iq = np.asarray(arr).astype(np.complex64)
        return iq

    # If 2-column last dim -> (I,Q)
    if arr.ndim == 2 and arr.shape[-1] == 2:
        I = arr[..., 0].astype(np.float32)
        Q = arr[..., 1].astype(np.float32)
        return I + 1j * Q

    # If 1D and even length -> assume interleaved I/Q
    if arr.ndim == 1 and (arr.size % 2 == 0):
        I = arr[0::2].astype(np.float32)
        Q = arr[1::2].astype(np.float32)
        return I + 1j * Q

    # If something unexpected, try last-ditch: flatten and interleave
    flat = arr.ravel()
    if flat.size % 2 == 0:
        I = flat[0::2].astype(np.float32)
        Q = flat[1::2].astype(np.float32)
        return I + 1j * Q

    raise ValueError(f"Unrecognized IQ layout for file: {path} with shape {arr.shape} dtype {arr.dtype}")

def window_fn(name, N):
    name = name.lower()
    if name == "hann":
        return np.hanning(N).astype(np.float32)
    if name == "hamming":
        return np.hamming(N).astype(np.float32)
    if name == "boxcar":
        return np.ones(N, dtype=np.float32)
    raise ValueError(f"Unsupported window type: {name}")

def stft_mag_db(xc, nfft=1024, hop=256, win="hann", eps=1e-10):
    """
    Simple STFT magnitude in dB for a complex baseband signal.
    Returns (S_db, times, freqs)
      S_db shape: (freq_bins, frames)
      freqs in Hz (baseband), times in seconds (assuming fs later)
    """
    x = np.asarray(xc, dtype=np.complex64)
    w = window_fn(win, nfft)
    # Number of frames
    if len(x) < nfft:
        # zero-pad if too short
        x = np.pad(x, (0, nfft - len(x)))
    n_frames = 1 + (len(x) - nfft) // hop if len(x) >= nfft else 1
    frames = np.lib.stride_tricks.as_strided(
        x,
        shape=(n_frames, nfft),
        strides=(x.strides[0]*hop, x.strides[0]),
        writeable=False
    )
    # Apply window
    frames = frames * w[None, :]
    # FFT (centered to put DC in middle)
    X = np.fft.fftshift(np.fft.fft(frames, n=nfft, axis=1), axes=1)
    # Power spectral density magnitude
    S = np.abs(X)**2
    # dB
    S_db = 10.0 * np.log10(S + eps)
    # transpose to (freq_bins, frames) for imshow
    S_db = S_db.T
    return S_db

def spectrogram_image(iq, fs, nfft=1024, hop=256, win="hann", vmin_db=None, vmax_db=None, ax=None):
    S_db = stft_mag_db(iq, nfft=nfft, hop=hop, win=win)
    # Frequency axis for display (baseband)
    freqs = np.linspace(-fs/2, fs/2, S_db.shape[0], endpoint=False)
    times = np.arange(S_db.shape[1]) * (hop / fs)

    if ax is None:
        ax = plt.gca()
    im = ax.imshow(
        S_db,
        origin="lower",
        aspect="auto",
        extent=[times[0], times[-1] if len(times)>1 else (len(times)/fs), freqs[0]/1e6, freqs[-1]/1e6],
        vmin=vmin_db,
        vmax=vmax_db,
        cmap="viridis",
    )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Freq (MHz)")
    return im

def read_train_list(train_txt, max_items=10):
    entries = []
    with open(train_txt, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            # Accept either whitespace or tab separated
            parts = line.split()
            if len(parts) < 2:
                continue
            relpath, label = parts[0], parts[1]
            # Defensive: strip commas/extra chars if any
            label = int(str(label).strip().replace(",", ""))
            entries.append((relpath, label))
            if len(entries) >= max_items:
                break
    return entries

def main():
    args = parse_args()
    entries = read_train_list(args.train_txt, max_items=args.max_items)
    if not entries:
        raise SystemExit("No entries found in train_txt.")

    nrows, ncols = 2, 5
    plt.figure(figsize=(18, 7))
    for i, (relpath, label) in enumerate(entries):
        fullpath = os.path.join(args.data_root, relpath)
        ax = plt.subplot(nrows, ncols, i + 1)
        try:
            iq = load_iq(fullpath)
            im = spectrogram_image(
                iq,
                fs=args.fs,
                nfft=args.nfft,
                hop=args.hop,
                win=args.win,
                vmin_db=args.vmin_db,
                vmax_db=args.vmax_db,
                ax=ax,
            )
            title_name = os.path.basename(relpath)
            ax.set_title(f"{title_name}\nlabel={label}", fontsize=10)
        except Exception as e:
            ax.text(0.5, 0.5, f"Error:\n{e}", ha="center", va="center", wrap=True)
            ax.set_axis_off()

    # Add a single colorbar for the whole figure
    cax = plt.axes([0.92, 0.15, 0.015, 0.7])
    # To make a colorbar we need a mappable; grab one from the last axes if present
    # Instead, create a dummy image to share the colormap/limits if none plotted
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors
    norm = None
    # Try to infer global vmin/vmax from plotted images
    # (If user supplied vmin_db/vmax_db these already constrain it)
    # For simplicity, create a ScalarMappable with the same cmap
    sm = cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])  # required for matplotlib < 3.7
    plt.colorbar(sm, cax=cax, label="Power (dB)")

    plt.tight_layout(rect=[0.03, 0.02, 0.9, 0.98])
    out_path = "spectrogram_grid.png"
    plt.savefig(out_path, dpi=200)
    print(f"Saved grid to {out_path}")
    plt.show()

if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] --train_txt TRAIN_TXT
                             [--data_root DATA_ROOT] [--fs FS] [--nfft NFFT]
                             [--hop HOP] [--win WIN] [--max_items MAX_ITEMS]
                             [--vmin_db VMIN_DB] [--vmax_db VMAX_DB]
ipykernel_launcher.py: error: the following arguments are required: --train_txt


SystemExit: 2

/usr/local/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
